# Decode: TPOT vs prompt length

1. Prefill results
2. Experiment design (shared by all models)
3. Qwen3-4B
   - Time at token k is linear in k
   - Intercept matches the prefill experiment
   - TPOT grows with prompt length
   - Total time: prefill vs decode
4. Not measured

Gemma 4 E2B and a model comparison will be added as sections once the Gemma sweep is run.

## Prefill results (from [`prefill_ttft.ipynb`](prefill_ttft.ipynb))

- vllm-mlx, Apple M1 8 GB, 4-bit weights. Prompt length set by noise headlines, 1.4k–21k tokens.
- TTFT stamped at token 2 (`prefill_only: true`, `max_tokens: 2`): prefill + 1 decode step.
- Fit `t = b·n + c·n²` (b: weight matmuls, c: attention):

| Model | b (ms/token) | c (µs/token²) | n* = b/c |
|---|---|---|---|
| Qwen3-4B | 6.2 | 0.21 | ~30k |
| Gemma 4 E2B | 0.86 | 0.009 | ~96k |

- WMT and AMZN agree within 1.6%: time depends on token count, not content.

This notebook measures the decode term and checks it against the prefill fit.

## Experiment design

Shared by all models. Same data, prompt, server and settings as the prefill experiment. Changes:

| | Prefill experiment | This experiment |
|---|---|---|
| `max_tokens` | 2 | 2, 16, 32, 64, 128 |
| Noise headlines | 50 … 1000 (AMZN 750) | 50, 200, 500 |
| Prompt tokens (WMT) | 1.4k–21k | 1.4k, 5.0k, 11.9k |

**Measurement**: streaming, `prefill_only: true`, `enable_thinking: true`. Each `max_tokens` value k is a separate request. Time stamped at the chunk carrying `finish_reason`: the arrival of token k. A row is valid only if `completion_tokens` = k. The column is named `ttft_s`; here it is the time at token k. Concurrency 1, continuous batching off, prefix cache off, 1 warm-up, 3 repeats per point.

**Model**: prefill P(n) produces token 1; each further token is one decode step.

`t(n, k) = P(n) + (k − 1)·TPOT(n)`

Fit `t = s·k + i` at each n: slope s = TPOT(n), intercept i = P(n) − TPOT(n). TPOT also grows with the context during decode (n + j at step j); over k ≤ 128 this adds < 25 ms and is ignored.

## Qwen3-4B

`mlx-community/Qwen3-4B-4bit`.

| Ticker | Signal headlines | Data | Config |
|---|---|---|---|
| WMT | 10 | [`results/tpot-decode-baseline-qwen3-wmt.csv`](results/tpot-decode-baseline-qwen3-wmt.csv) | [`configs/tpot-decode-baseline-qwen3-wmt.yaml`](configs/tpot-decode-baseline-qwen3-wmt.yaml) |
| AMZN | 7 | [`results/tpot-decode-baseline-qwen3-amzn.csv`](results/tpot-decode-baseline-qwen3-amzn.csv) | [`configs/tpot-decode-baseline-qwen3-amzn.yaml`](configs/tpot-decode-baseline-qwen3-amzn.yaml) |

### Time at token k is linear in k

Fit uses all 15 rows per prompt length (5 values of k × 3 repeats); ± is the standard error of the fit.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUNS = {"WMT": "results/tpot-decode-baseline-qwen3-wmt.csv", "AMZN": "results/tpot-decode-baseline-qwen3-amzn.csv"}


def line_fit(k, t):
    X = np.c_[k, np.ones_like(k)]
    coef = np.linalg.lstsq(X, t, rcond=None)[0]
    r = t - X @ coef
    se = np.sqrt(np.diag(r @ r / (len(t) - 2) * np.linalg.inv(X.T @ X)))
    return coef, se


def decode_fits(path):
    df = pd.read_csv(path)
    df = df[df.ok & (df.completion_tokens == df.max_tokens)]
    rows = []
    for noise, g in df.groupby("noise_headlines"):
        (s, i), (s_se, i_se) = line_fit(g.max_tokens.to_numpy(float), g.ttft_s.to_numpy())
        rows.append({"noise": noise, "n": g.prompt_tokens.median(), "rows": len(g),
                     "tpot_ms": s * 1e3, "tpot_se_ms": s_se * 1e3, "intercept_s": i, "intercept_se_s": i_se})
    return df, pd.DataFrame(rows).set_index("noise")


data, tpot = {}, {}
for k, p in RUNS.items():
    data[k], tpot[k] = decode_fits(p)
pd.concat(tpot, names=["ticker"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (noise, row) in zip(axes, tpot["WMT"].iterrows()):
    for (name, df), style in zip(data.items(), [dict(fmt="ko"), dict(fmt="s", mfc="none", mec="C3", ms=8)]):
        m = df[df.noise_headlines == noise].groupby("max_tokens").ttft_s
        ax.plot(m.median().index, m.median() - row.intercept_s, style.pop("fmt"), label=name, **style)
    k = np.array([0, 130])
    ax.plot(k, row.tpot_ms / 1e3 * k, label=f"fit (WMT): {row.tpot_ms:.1f} ms/token")
    ax.set(title=f"n ≈ {row.n:,.0f} prompt tokens", xlabel="k (output tokens)", ylim=(0, None))
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("time at token k − WMT intercept (s)")
fig.tight_layout();

### Intercept matches the prefill experiment

Both experiments stamp a time at token 2. From this experiment: `t(n, 2) = i + 2·s`. From the prefill experiment: `b·n + c·n²`, refitted below from its committed results. Separate runs, 5 days (AMZN) and 12 days (WMT) apart.

In [ ]:
PREFILL = {"WMT": "results/ttft-prefill-baseline.csv", "AMZN": "results/ttft-prefill-baseline-amzn.csv"}


def prefill_fit(path):
    df = pd.read_csv(path).dropna(subset=["ttft_s"])
    m = df[df.ok].groupby("noise_headlines")[["prompt_tokens", "ttft_s"]].median()
    n, t = m.prompt_tokens.to_numpy(), m.ttft_s.to_numpy()
    return np.linalg.lstsq(np.c_[n, n**2] / t[:, None], np.ones_like(t), rcond=None)[0]


rows = []
for k, path in PREFILL.items():
    b, c = prefill_fit(path)
    for noise, r in tpot[k].iterrows():
        t2 = r.intercept_s + 2 * r.tpot_ms / 1e3
        pf = b * r.n + c * r.n**2
        rows.append({"ticker": k, "noise": noise, "n": r.n, "t2_decode_fit_s": t2, "prefill_fit_s": pf,
                     "diff_%": 100 * (t2 / pf - 1), "P_s": r.intercept_s + r.tpot_ms / 1e3})
pd.DataFrame(rows).set_index(["ticker", "noise"]).round(3)

- The two experiments agree within −1.2% to +2.2% at the same prompt length. Run-to-run drift in the prefill experiment was 0.2–2%.
- `P_s` = i + s: prefill alone, time to token 1.

### TPOT grows with prompt length: the KV cache is read every step

Each decode step reads all weights and the whole KV cache.

| Per decode step | Bytes |
|---|---|
| Weights (`model.safetensors`, 4-bit + fp16 scales/biases; embeddings tied to the output projection, so read in full) | 2.1 GB |
| KV cache per context token: 36 layers × 8 KV heads × 128 × 2 (K, V) × 2 bytes | 144 KiB |

Model: `TPOT(n) = (W + n·kv) / BW_eff`, i.e. `a + β·n` with a = W / BW_eff, β = kv / BW_eff.

In [ ]:
W, KV = 2.1e9, 36 * 8 * 128 * 2 * 2
both = pd.concat(tpot, names=["ticker"]).reset_index()
beta, a = np.polyfit(both.n, both.tpot_ms / 1e3, 1)
both["bytes_GB"] = (W + both.n * KV) / 1e9
both["bw_eff_GB_s"] = both.bytes_GB / (both.tpot_ms / 1e3)
print(f"TPOT ≈ {a * 1e3:.1f} ms + {beta * 1e6:.2f} µs·n")
print(f"BW_eff from a: {W / a / 1e9:.0f} GB/s   from β: {KV / beta / 1e9:.0f} GB/s   M1 peak: ~68 GB/s")

x = np.linspace(0, 1.05 * both.n.max(), 100)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, (a + beta * x) * 1e3, label=f"fit: {a * 1e3:.1f} ms + {beta * 1e6:.2f} µs·n")
ax.plot(x, W / 68e9 * 1e3 + KV / 68e9 * 1e3 * x, "--", label="bandwidth floor at 68 GB/s")
for (name, g), style in zip(both.groupby("ticker", sort=False), [dict(fmt="ko"), dict(fmt="s", mfc="none", mec="C3", ms=9)]):
    ax.errorbar(g.n, g.tpot_ms, yerr=g.tpot_se_ms, label=name, **style)
ax.set(xlabel="Prompt length (tokens)", ylabel="TPOT (ms/token)", xlim=(0, None), ylim=(0, None))
ax.grid(alpha=0.3)
ax.legend()
both[["ticker", "noise", "n", "tpot_ms", "bytes_GB", "bw_eff_GB_s"]].round(2)

- Weights dominate the bytes at 1.4k tokens; at 11.9k the KV cache (1.75 GB) is close to the weights.
- Effective bandwidth is 50–52 GB/s at all lengths, ~75% of the 68 GB/s peak: decode is memory-bandwidth-bound.
- The intercept a implies 50 GB/s for the weight read and the slope β implies 55 GB/s for the KV read: both reads run at similar efficiency.
- WMT slope between points rises from 2.45 to 2.90 µs/token; three points do not separate curvature from noise.
- AMZN prompts are ~48 tokens longer and fall on the same line: TPOT depends on token count, not content.

### Total time: prefill vs decode

`t(n, k) = P(n) + (k − 1)·TPOT(n)` with P(n) from the prefill fit and TPOT(n) from the fit above.

In [ ]:
b, c = prefill_fit(PREFILL["WMT"])
rows = []
for n in [1_000, 5_000, 12_000, 20_000]:
    for k in [128, 512, 2048]:
        tp = a + beta * n
        P = b * n + c * n**2 - tp
        rows.append({"n": n, "k": k, "prefill_s": P, "decode_s": (k - 1) * tp, "total_s": P + (k - 1) * tp,
                     "prefill_%": 100 * P / (P + (k - 1) * tp)})
pd.DataFrame(rows).set_index(["n", "k"]).round(1)

- Extrapolated: k > 128 and n = 20k are outside this sweep. Ignores TPOT growth with k (adds β·k²/2: ~6 s at k = 2048).
- Prefill is more than half the total up to k ≈ 650 at n = 5k and k ≈ 1,400 at n = 12k.
- Prefill cost is per request and n-dependent; prefix caching removes it for a shared prefix. Decode cost is per token and mostly fixed by weight bytes.

## Not measured

- Concurrency > 1: batched decode amortises the weight read across requests.
- Gemma 4 E2B decode.
- Run-to-run drift: one run per ticker; both stamped `3cd2e4e-dirty`.